In [1]:
# ETAPA 3 — BANCO DE DADOS

from pathlib import Path
import pandas as pd
import sqlite3
import json

# Localização dos arquivos
PASTA_PROJETO = Path.cwd()

if PASTA_PROJETO.name == "notebooks":
    PASTA_PROJETO = PASTA_PROJETO.parent

PASTA_DADOS = PASTA_PROJETO / "dados"
PASTA_BANCO = PASTA_PROJETO / "banco"

PASTA_BANCO.mkdir(exist_ok=True)

# Carregamento das sete tabelas
hotels = pd.read_json(PASTA_DADOS / "hotels.jsonl", lines=True)
room_types = pd.read_json(PASTA_DADOS / "room_types.jsonl", lines=True)
daily_market = pd.read_json(PASTA_DADOS / "daily_market.jsonl", lines=True)
weekly_performance = pd.read_json(PASTA_DADOS / "weekly_performance.jsonl", lines=True)
rate_decisions = pd.read_json(PASTA_DADOS / "rate_decisions.jsonl", lines=True)
reviews = pd.read_json(PASTA_DADOS / "reviews.jsonl", lines=True)
review_scenarios = pd.read_json(PASTA_DADOS / "review_scenarios.jsonl", lines=True)

tabelas = {
    "hotels": hotels,
    "room_types": room_types,
    "daily_market": daily_market,
    "weekly_performance": weekly_performance,
    "rate_decisions": rate_decisions,
    "reviews": reviews,
    "review_scenarios": review_scenarios
}

# Conferência
for nome, df in tabelas.items():
    print(f"{nome}: {df.shape[0]} linhas e {df.shape[1]} colunas")

hotels: 14 linhas e 17 colunas
room_types: 42 linhas e 10 colunas
daily_market: 10192 linhas e 56 colunas
weekly_performance: 1456 linhas e 41 colunas
rate_decisions: 1442 linhas e 24 colunas
reviews: 12077 linhas e 26 colunas
review_scenarios: 11 linhas e 7 colunas


In [2]:
# CRIAÇÃO DO BANCO DE DADOS SQLITE

# Caminho do arquivo do banco
CAMINHO_BANCO = PASTA_BANCO / "port_alder.db"

# Conectar ao banco
conexao = sqlite3.connect(CAMINHO_BANCO)

# Ativar a verificação das chaves estrangeiras
conexao.execute("PRAGMA foreign_keys = ON")

# Definição das chaves primárias e estrangeiras
configuracoes = {
    "hotels": {
        "pk": "hotel_id"
    },
    "room_types": {
        "pk": "hotel_id, room_type_id",
        "unique": ["hotel_id, room_type"]
    },
    "daily_market": {
        "pk": "hotel_id, date"
    },
    "weekly_performance": {
        "pk": "hotel_id, week_start"
    },
    "rate_decisions": {
        "pk": "decision_id",
        "fk": ["FOREIGN KEY (hotel_id) REFERENCES hotels(hotel_id)"]
    },
    "reviews": {
        "pk": "review_id",
        "fk": [
            "FOREIGN KEY (hotel_id) REFERENCES hotels(hotel_id)",
            "FOREIGN KEY (scenario_id) REFERENCES review_scenarios(scenario_id)",
            "FOREIGN KEY (hotel_id, room_type) REFERENCES room_types(hotel_id, room_type)"
        ]
    },
    "review_scenarios": {
        "pk": "scenario_id"
    }
}

# Ordem de carregamento respeitando os relacionamentos
ordem = [
    "hotels",
    "room_types",
    "review_scenarios",
    "daily_market",
    "weekly_performance",
    "rate_decisions",
    "reviews"
]

# Criar as tabelas e carregar os dados
for nome in ordem:

    df = tabelas[nome].copy()
    config = configuracoes[nome]

    # Converter listas em JSON e datas em texto
    for coluna in df.columns:
        if df[coluna].dtype == "object":
            df[coluna] = df[coluna].apply(
                lambda valor: json.dumps(valor, ensure_ascii=False)
                if isinstance(valor, list) else valor
            )

    # Identificar os tipos de dados
    definicoes = []

    for coluna in df.columns:
        tipo = df[coluna].dtype

        if pd.api.types.is_integer_dtype(tipo):
            tipo_sql = "INTEGER"
        elif pd.api.types.is_float_dtype(tipo):
            tipo_sql = "REAL"
        else:
            tipo_sql = "TEXT"

        definicoes.append(f'"{coluna}" {tipo_sql}')

    # Adicionar chave primária
    definicoes.append(f'PRIMARY KEY ({config["pk"]})')

    # Adicionar restrições UNIQUE
    for restricao in config.get("unique", []):
        definicoes.append(f"UNIQUE ({restricao})")

    # Adicionar chaves estrangeiras
    definicoes.extend(config.get("fk", []))

    # Criar tabela
    comando = f"""
    CREATE TABLE "{nome}" (
        {", ".join(definicoes)}
    )
    """

    conexao.execute(comando)

    # Inserir os registros
    df.to_sql(nome, conexao, if_exists="append", index=False)

    print(f"Tabela {nome} criada e carregada: {len(df)} registros")

conexao.commit()

print("\nBanco de dados criado com sucesso!")
print("Arquivo:", CAMINHO_BANCO)

Tabela hotels criada e carregada: 14 registros
Tabela room_types criada e carregada: 42 registros
Tabela review_scenarios criada e carregada: 11 registros
Tabela daily_market criada e carregada: 10192 registros
Tabela weekly_performance criada e carregada: 1456 registros
Tabela rate_decisions criada e carregada: 1442 registros
Tabela reviews criada e carregada: 12077 registros

Banco de dados criado com sucesso!
Arquivo: c:\Users\luiza\Documents\Projetos\projeto-integrador-ciencia-de-dados\banco\port_alder.db


In [3]:
# TESTES DE INTEGRIDADE DO BANCO

# 1. Conferir quantidade de registros em cada tabela
print("QUANTIDADE DE REGISTROS POR TABELA")

for nome in ordem:
    resultado = conexao.execute(
        f'SELECT COUNT(*) FROM "{nome}"'
    ).fetchone()[0]

    print(f"{nome}: {resultado}")


# 2. Verificar chaves estrangeiras
print("\nVERIFICAÇÃO DAS CHAVES ESTRANGEIRAS")

erros_fk = conexao.execute(
    "PRAGMA foreign_key_check"
).fetchall()

if len(erros_fk) == 0:
    print("Nenhum problema encontrado nas chaves estrangeiras.")
else:
    print("Problemas encontrados:")
    print(erros_fk)


# 3. Verificar integridade geral do banco
print("\nINTEGRIDADE GERAL")

integridade = conexao.execute(
    "PRAGMA integrity_check"
).fetchone()[0]

print(integridade)

QUANTIDADE DE REGISTROS POR TABELA
hotels: 14
room_types: 42
review_scenarios: 11
daily_market: 10192
weekly_performance: 1456
rate_decisions: 1442
reviews: 12077

VERIFICAÇÃO DAS CHAVES ESTRANGEIRAS
Nenhum problema encontrado nas chaves estrangeiras.

INTEGRIDADE GERAL
ok


In [6]:
consulta_1 = """
SELECT
    h.hotel_id,
    h.name AS hotel,
    strftime('%Y', d.date) AS ano,
    ROUND(SUM(d.room_revenue), 2) AS receita_total
FROM daily_market AS d
JOIN hotels AS h
    ON d.hotel_id = h.hotel_id
GROUP BY
    h.hotel_id,
    h.name,
    ano
ORDER BY
    ano,
    receita_total DESC;
"""

resultado_1 = pd.read_sql_query(consulta_1, conexao)

display(resultado_1)

,hotel_id,hotel,ano,receita_total
0,H03,Marisol Bay Resort,2027,22068551.84
1,H06,Seabright Grand,2027,13966098.53
2,H01,Alder & Quay Hotel,2027,10447256.42
3,H04,Harborlight Suites,2027,9541961.54
4,H05,The Copper Meridian,2027,8945014.07
5,H10,Kestrel Park Hotel,2027,7176843.38
6,H14,Willowmere Lodge,2027,5629193.20
7,H09,Cedar & Salt Hotel,2027,5424371.03
8,H13,Paloma Court Hotel,2027,5067839.92
9,H02,The Lantern House,2027,3922279.31


In [7]:
consulta_2 = """
SELECT
    tier,
    season,
    ROUND(AVG(occupancy) * 100, 2) AS ocupacao_media_pct,
    ROUND(AVG(adr), 2) AS diaria_media
FROM daily_market
GROUP BY tier, season
ORDER BY tier, season;
"""

resultado_2 = pd.read_sql_query(consulta_2, conexao)

display(resultado_2)

,tier,season,ocupacao_media_pct,diaria_media
0,boutique,autumn,74.97,205.37
1,boutique,spring,74.97,204.03
2,boutique,summer,86.78,253.95
3,boutique,winter,66.17,179.76
4,economy,autumn,60.97,88.68
5,economy,spring,61.23,88.31
6,economy,summer,71.48,108.97
7,economy,winter,53.82,78.46
8,luxury,autumn,71.48,357.39
9,luxury,spring,71.46,355.56


In [8]:
consulta_3 = """
SELECT
    h.hotel_id,
    h.name AS hotel,
    COUNT(r.review_id) AS quantidade_avaliacoes,
    ROUND(AVG(r.rating), 2) AS avaliacao_media
FROM hotels AS h
LEFT JOIN reviews AS r
    ON h.hotel_id = r.hotel_id
GROUP BY
    h.hotel_id,
    h.name
ORDER BY avaliacao_media DESC;
"""

resultado_3 = pd.read_sql_query(consulta_3, conexao)

display(resultado_3)

,hotel_id,hotel,quantidade_avaliacoes,avaliacao_media
0,H05,The Copper Meridian,677,3.57
1,H03,Marisol Bay Resort,1538,3.55
2,H01,Alder & Quay Hotel,1204,3.51
3,H02,The Lantern House,420,3.45
4,H11,The Foxglove,344,3.39
5,H04,Harborlight Suites,1008,3.33
6,H06,Seabright Grand,1733,3.32
7,H14,Willowmere Lodge,679,3.25
8,H13,Paloma Court Hotel,829,3.24
9,H09,Cedar & Salt Hotel,963,3.21


In [9]:
consulta_4 = """
SELECT
    tier,
    ROUND(AVG(room_revenue), 2) AS receita_media,
    ROUND(SUM(room_revenue), 2) AS receita_total,
    COUNT(*) AS quantidade_registros
FROM daily_market
GROUP BY tier
ORDER BY receita_total DESC;
"""

resultado_4 = pd.read_sql_query(consulta_4, conexao)

display(resultado_4)

,tier,receita_media,receita_total,quantidade_registros
0,upscale,27320.76,79558052.83,2912
1,luxury,42801.68,62319247.88,1456
2,midscale,16276.40,35547651.07,2184
3,boutique,7568.68,16529991.52,2184
4,economy,6047.19,8804701.90,1456


In [10]:
consulta_5 = """
SELECT
    demand_event,
    COUNT(*) AS quantidade_registros,
    ROUND(AVG(occupancy) * 100, 2) AS ocupacao_media_pct,
    ROUND(AVG(adr), 2) AS diaria_media
FROM daily_market
WHERE demand_event IS NOT NULL
  AND TRIM(demand_event) <> ''
GROUP BY demand_event
ORDER BY ocupacao_media_pct DESC;
"""

resultado_5 = pd.read_sql_query(consulta_5, conexao)

display(resultado_5)

,demand_event,quantidade_registros,ocupacao_media_pct,diaria_media
0,Regatta Week,196,91.14,299.51
1,Alder Music Festival,84,86.16,267.69
2,Autumn Design Fair,112,78.01,225.84
3,New Year Harbour,70,75.34,231.19
4,Harbour Tech Summit,112,70.66,206.58
5,Spring Food Fair,84,68.17,193.14


In [13]:
consulta_6 = """
SELECT
    s.scenario_id,
    s.label AS cenario,
    COUNT(r.review_id) AS quantidade_avaliacoes,
    ROUND(AVG(r.rating), 2) AS avaliacao_media
FROM review_scenarios AS s
LEFT JOIN reviews AS r
    ON s.scenario_id = r.scenario_id
GROUP BY
    s.scenario_id,
    s.label
ORDER BY avaliacao_media DESC;
"""

resultado_6 = pd.read_sql_query(consulta_6, conexao)

display(resultado_6)

,scenario_id,cenario,quantidade_avaliacoes,avaliacao_media
0,delight,Stay exceeded expectations,1840,4.84
1,location,Location and access,1219,4.58
2,service_recovery,Service recovery,1206,4.33
3,group_trip,Group or family stay,472,4.00
4,breakfast,Breakfast and food,1023,3.73
5,checkin_wait,Check-in and queue,1006,2.81
6,wifi,Wifi and connectivity,765,2.74
7,room_condition,Room condition,1401,2.48
8,noise,Noise and sleep,1250,2.42
9,cleanliness,Cleanliness,868,1.69


In [14]:
consulta_7 = """
SELECT
    h.hotel_id,
    h.name AS hotel,
    COUNT(r.review_id) AS total_avaliacoes,
    COUNT(r.response_hours) AS avaliacoes_respondidas,
    ROUND(
        100.0 * COUNT(r.response_hours) / COUNT(r.review_id),
        2
    ) AS percentual_respondido,
    ROUND(AVG(r.response_hours), 2) AS tempo_medio_resposta_horas
FROM hotels AS h
LEFT JOIN reviews AS r
    ON h.hotel_id = r.hotel_id
GROUP BY
    h.hotel_id,
    h.name
ORDER BY percentual_respondido DESC;
"""

resultado_7 = pd.read_sql_query(consulta_7, conexao)

display(resultado_7)

,hotel_id,hotel,total_avaliacoes,avaliacoes_respondidas,percentual_respondido,tempo_medio_resposta_horas
0,H12,Ridgeline Motor Inn,506,388,76.68,21.09
1,H08,Northgate Inn,718,546,76.04,20.79
2,H10,Kestrel Park Hotel,1169,858,73.40,20.88
3,H09,Cedar & Salt Hotel,963,681,70.72,21.17
4,H13,Paloma Court Hotel,829,586,70.69,21.99
5,H04,Harborlight Suites,1008,712,70.63,20.91
6,H07,The Wren Rooms,289,204,70.59,22.09
7,H14,Willowmere Lodge,679,478,70.40,21.49
8,H06,Seabright Grand,1733,1202,69.36,20.97
9,H02,The Lantern House,420,278,66.19,20.51
